In [1]:
import os
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data, Model
from azure.identity import InteractiveBrowserCredential
from azure.storage.blob import BlobServiceClient

load_dotenv()

print("⚙️  Setup Azure ML Client...")

# Gunakan tenant ID yang benar dari error message
TENANT_ID = "c7dc0889-56ce-4b24-9738-8b15a52a516a"

credential = InteractiveBrowserCredential(tenant_id=TENANT_ID)

ml_client = MLClient(
    credential=credential,
    subscription_id=os.getenv("AZURE_ML_SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("AZURE_ML_RESOURCE_GROUP"),
    workspace_name=os.getenv("AZURE_ML_WORKSPACE_NAME")
)

print(f"✅ ML Client connected!")
print(f"   Workspace : {os.getenv('AZURE_ML_WORKSPACE_NAME')}")
print(f"   Tenant ID : {TENANT_ID}")

python-dotenv could not parse statement starting at line 1


⚙️  Setup Azure ML Client...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


✅ ML Client connected!
   Workspace : sipadi-workspace
   Tenant ID : c7dc0889-56ce-4b24-9738-8b15a52a516a


In [2]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

print("📦 Registering datasets ke Azure ML...")

PROCESSED_DIR = Path("../data/processed")

# Register master dataset
data_asset = Data(
    name="sipadi-master-dataset",
    version="1",
    description="Master dataset SiPADI - 7 sumber data, 51 fitur, 1624 records",
    path=str(PROCESSED_DIR / "master_dataset.csv"),
    type=AssetTypes.URI_FILE,
    tags={
        "project": "sipadi",
        "source": "BPS,BMKG,NOAA,Sentinel,Kementan,Kemendag",
        "records": "1624",
        "features": "51"
    }
)

try:
    registered_data = ml_client.data.create_or_update(data_asset)
    print(f"✅ Dataset registered!")
    print(f"   Name    : {registered_data.name}")
    print(f"   Version : {registered_data.version}")
    print(f"   URI     : {registered_data.path}")
except Exception as e:
    print(f"⚠️  Dataset registration: {e}")

📦 Registering datasets ke Azure ML...


d:\sipadi\venv\Lib\site-packages\msal\oauth2cli\oauth2.py:408: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


⚠️  Dataset registration: (UserError) A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Code: UserError
Message: A data version with this name and version already exists. If you are trying to create a new data version, use a different name or version. If you are trying to update an existing data version, the existing asset's data uri cannot be changed. Only tags, description, and isArchived can be updated.
Additional Information:Type: ComponentName
Info: {
    "value": "managementfrontend"
}Type: Correlation
Info: {
    "value": {
        "operation": "7f5fbce80990a03990af6b138bd47e71",
        "request": "4b7823e47b6921e1"
    }
}Type: Environment
Info: {
    "value": "southeastasia"
}Type: Location
Info: {
    "value": "southeastasia"
}Type:

In [9]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

print("🤖 Registering models ke Azure ML...")

MODELS_DIR = Path("../models")

models_to_register = [
    {
        "name": "sipadi-productivity-model",
        "path": str(MODELS_DIR / "productivity_model.pkl"),
        "description": "XGBoost Regressor - R2=0.886 MAPE=2.25%",
        "tags": {"algorithm": "XGBoost", "r2": "0.886", "mape": "2.25"}
    },
    {
        "name": "sipadi-risk-classifier",
        "path": str(MODELS_DIR / "risk_classifier.pkl"),
        "description": "XGBoost Classifier - F1=0.833 AUC=0.967",
        "tags": {"algorithm": "XGBoost", "f1": "0.833", "auc": "0.967"}
    },
    {
        "name": "sipadi-price-forecaster",
        "path": str(MODELS_DIR / "price_forecaster.pkl"),
        "description": "SARIMA - MAPE=1.16%",
        "tags": {"algorithm": "SARIMA", "mape": "1.16"}
    }
]

for m in models_to_register:
    try:
        model = Model(
            name=m["name"],
            path=m["path"],
            description=m["description"],
            tags=m["tags"]
        )
        registered = ml_client.models.create_or_update(model)
        print(f"✅ {registered.name} v{registered.version}")
    except Exception as e:
        print(f"⚠️  {m['name']}: {str(e)[:80]}")

🤖 Registering models ke Azure ML...
✅ sipadi-productivity-model v4
✅ sipadi-risk-classifier v4
✅ sipadi-price-forecaster v4


In [13]:
pip install "azure-storage-blob>=12.28.0" --upgrade

  Using cached azure_storage_blob-12.28.0-py3-none-any.whl.metadata (26 kB)
Using cached azure_storage_blob-12.28.0-py3-none-any.whl (431 kB)
  Attempting uninstall: azure-storage-blob
    Found existing installation: azure-storage-blob 12.19.0
    Uninstalling azure-storage-blob-12.19.0:
      Successfully uninstalled azure-storage-blob-12.19.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azureml-mlflow 1.60.0 requires azure-storage-blob<=12.19.0,>=12.5.0, but you have azure-storage-blob 12.28.0 which is incompatible.


In [4]:
print("📊 Logging experiment metrics ke Azure ML...")
# Kita gunakan MLflow yang terintegrasi dengan Azure ML
import mlflow

# Set tracking URI ke Azure ML
mlflow_tracking_uri = ml_client.workspaces.get(
    os.getenv("AZURE_ML_WORKSPACE_NAME")
).mlflow_tracking_uri

mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment("sipadi-model-training")

print(f"✅ MLflow tracking URI: {mlflow_tracking_uri}")

# Log Model 1 - Regression
with mlflow.start_run(run_name="xgboost-productivity-regression"):
    mlflow.log_params({
        "algorithm": "XGBoost",
        "n_features": 36,
        "train_size": 1299,
        "test_size": 325,
        "cv_folds": 5
    })
    mlflow.log_metrics({
        "rmse": 0.1577,
        "mae": 0.1259,
        "r2": 0.8863,
        "mape": 2.25,
        "cv_rmse": 0.16
    })
    mlflow.set_tags({
        "model_type": "regression",
        "target": "produktivitas_ton_per_ha",
        "project": "sipadi"
    })
    print("✅ Run 1 logged: XGBoost Regression")

# Log Model 2 - Classification
with mlflow.start_run(run_name="xgboost-risk-classification"):
    mlflow.log_params({
        "algorithm": "XGBoost",
        "n_features": 36,
        "class_weight": "scale_pos_weight",
        "cv_folds": 5
    })
    mlflow.log_metrics({
        "f1_score": 0.8333,
        "precision": 0.7576,
        "recall": 0.9259,
        "roc_auc": 0.9665,
        "cv_f1": 0.82
    })
    mlflow.set_tags({
        "model_type": "classification",
        "target": "risiko_gagal_panen",
        "project": "sipadi"
    })
    print("✅ Run 2 logged: XGBoost Classification")

# Log Model 3 - Time Series
with mlflow.start_run(run_name="sarima-price-forecasting"):
    mlflow.log_params({
        "algorithm": "SARIMA",
        "order": "(1,1,1)",
        "seasonal_order": "(1,1,1,12)",
        "train_size": 134,
        "test_size": 34
    })
    mlflow.log_metrics({
        "rmse": 216.36,
        "mae": 154.59,
        "mape": 1.16
    })
    mlflow.set_tags({
        "model_type": "timeseries",
        "target": "harga_beras_medium_per_kg",
        "forecast_horizon": "12",
        "project": "sipadi"
    })
    print("✅ Run 3 logged: SARIMA Time Series")

print("\n✅ Semua experiment metrics ter-log di Azure ML!")

📊 Logging experiment metrics ke Azure ML...


2026/04/26 11:53:14 INFO mlflow.tracking.fluent: Experiment with name 'sipadi-model-training' does not exist. Creating a new experiment.


✅ MLflow tracking URI: azureml://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/9f74d9b7-3a90-4f07-a4b3-6808c0bf37a8/resourceGroups/sipadi-rg/providers/Microsoft.MachineLearningServices/workspaces/sipadi-workspace
✅ Run 1 logged: XGBoost Regression
🏃 View run xgboost-productivity-regression at: https://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/9f74d9b7-3a90-4f07-a4b3-6808c0bf37a8/resourceGroups/sipadi-rg/providers/Microsoft.MachineLearningServices/workspaces/sipadi-workspace/#/experiments/4474dd0c-ab5a-43b7-99d5-24d45cb3903b/runs/7ed73b5f-4083-4cc7-adf8-f8417d68469d
🧪 View experiment at: https://southeastasia.api.azureml.ms/mlflow/v2.0/subscriptions/9f74d9b7-3a90-4f07-a4b3-6808c0bf37a8/resourceGroups/sipadi-rg/providers/Microsoft.MachineLearningServices/workspaces/sipadi-workspace/#/experiments/4474dd0c-ab5a-43b7-99d5-24d45cb3903b
✅ Run 2 logged: XGBoost Classification
🏃 View run xgboost-risk-classification at: https://southeastasia.api.azureml.ms/mlflow/v2.0/subsc

In [11]:
print("=" * 60)
print("🔍 VERIFIKASI AZURE ML WORKSPACE")
print("=" * 60)

# Cek registered models
print("\n📦 Registered Models:")
try:
    for model in ml_client.models.list():
        print(f"   ✅ {model.name} v{model.version} — {model.description[:50]}...")
except Exception as e:
    print(f"   ⚠️  {e}")

# Cek experiments
print("\n📊 MLflow Experiments:")
try:
    experiments = mlflow.search_experiments()
    for exp in experiments:
        print(f"   ✅ {exp.name}")
        runs = mlflow.search_runs(exp.experiment_id)
        for _, run in runs.iterrows():
            print(f"      → {run['tags.mlflow.runName']}")
except Exception as e:
    print(f"   ⚠️  {e}")

print("\n" + "=" * 60)
print("✅ AZURE ML PIPELINE SELESAI!")
print("=" * 60)
print(f"""
Ringkasan Azure Services yang Digunakan:
   ✅ Azure Blob Storage   — Data lake (7 datasets)
   ✅ Azure ML Workspace   — Model registry & experiment tracking  
   ✅ Azure AI Language    — Sentiment analysis (siap dipakai)
   ✅ Azure Maps           — Visualisasi spasial (siap dipakai)
   ✅ Azure Static Web Apps — Dashboard deployment (next step)
""")

🔍 VERIFIKASI AZURE ML WORKSPACE

📦 Registered Models:
   ⚠️  'NoneType' object is not subscriptable

📊 MLflow Experiments:
   ✅ sipadi-model-training
      → xgboost-productivity-regression
      → xgboost-risk-classification
      → sarima-price-forecasting

✅ AZURE ML PIPELINE SELESAI!

Ringkasan Azure Services yang Digunakan:
   ✅ Azure Blob Storage   — Data lake (7 datasets)
   ✅ Azure ML Workspace   — Model registry & experiment tracking  
   ✅ Azure AI Language    — Sentiment analysis (siap dipakai)
   ✅ Azure Maps           — Visualisasi spasial (siap dipakai)
   ✅ Azure Static Web Apps — Dashboard deployment (next step)

